<a href="https://colab.research.google.com/github/LimJavier/ISYS6014_CloudCore_Churn_Prediction_Dashboard/blob/main/Group_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

!pip -q install streamlit

Saving cloudcore_churn_predictions.csv to cloudcore_churn_predictions.csv
Dataset shape: (282, 12)

Column names:
['churn_flag', 'industry', 'customer_id', 'company_name', 'Logistic Regression', 'Logistic Regression (0)', 'Logistic Regression (1)', 'tenure_months', 'monthly_recurring_revenue', 'product_count', 'support_tier', 'region']


,churn_flag,industry,customer_id,company_name,Logistic Regression,Logistic Regression (0),Logistic Regression (1),tenure_months,monthly_recurring_revenue,product_count,support_tier,region
0,0 1,Education Energy Finance Healthcare Logistics ...,string,string,0 1,continuous,continuous,continuous,continuous,continuous,Basic Enterprise Premium Standard,San\ Francisco Singapore Sydney
1,class,meta,meta,meta,meta,meta,meta,NaN,NaN,NaN,NaN,NaN
2,0,Education,C0002,Laurel Institute,0,0.582087,0.417913,7,1360,1,Standard,Sydney
3,0,Energy,C0005,Quartz Utilities,1,0.452296,0.547704,19,2160,1,Basic,Sydney
4,1,Technology,C0012,Onyx Digital,1,0.163428,0.836572,18,1560,1,Basic,San Francisco


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.9 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

st.set_page_config(
    page_title="CloudCore Retention Review Queue",
    page_icon="📊",
    layout="wide"
)

st.title("CloudCore Retention Review Queue")
st.caption("Ranked customers for human review. This is decision support, not an automatic decision.")

# ---------- Column names (from the Orange Predictions export) ----------
PROB = "Logistic Regression (1)"   # probability of class 1
PRED = "Logistic Regression"       # predicted class at Orange's default 0.5
NUM = ["tenure_months", "monthly_recurring_revenue", "product_count"]
CAT = ["support_tier", "region"]


@st.cache_data
def load_data():
    df = pd.read_csv("cloudcore_churn_predictions.csv", skiprows=[1, 2])
    df.columns = df.columns.str.strip()
    for col in ["churn_flag", PRED, "Logistic Regression (0)", PROB] + NUM:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


try:
    df = load_data()
except Exception as e:
    st.error(f"Could not load the CSV file: {e}")
    st.stop()

required = ["customer_id", "company_name", "churn_flag", PRED, PROB] + NUM + CAT
missing = [c for c in required if c not in df.columns]
if missing:
    st.error(f"Missing required columns: {missing}")
    st.stop()

df = df.dropna(subset=[PROB, "churn_flag"]).copy()

# ---------- Rank every customer by churn probability ----------
df["Churn Probability (%)"] = (df[PROB] * 100).round(1)
df["Actual churn (history)"] = df["churn_flag"].map(
    {1: "Churned", 0: "Did not churn"}
)
ranked = df.sort_values(PROB, ascending=False).reset_index(drop=True)
ranked.insert(0, "Rank", ranked.index + 1)

# ---------- Sidebar controls ----------
st.sidebar.header("Review capacity")
n = st.sidebar.slider("Customers the team can review per week", 5, 25, 10)
st.sidebar.caption(
    "Support lead's estimate: about 10-15 quick reviews, or 5-8 deeper reviews, per week."
)
ranked["In weekly queue"] = ranked["Rank"] <= n

st.sidebar.header("Back-test")
reveal = st.sidebar.checkbox(
    "Show actual churn (history)", value=False,
    help="Adds what actually happened to each customer in the historical data, so you "
         "can check how well the ranking worked. The support team would not have this."
)

if reveal:
    st.info(
        "Back-test view: the model's churn probabilities are shown next to what actually "
        "happened in the historical data. 'Actual churn' is CloudCore's historical churn "
        "label, which also includes customers flagged as at-risk."
    )
else:
    st.info(
        "Customers are ranked by the model's churn probability. The highest-ranked "
        "customers form this week's review queue."
    )

total = len(ranked)
total_pos = int(ranked["churn_flag"].sum())
queue = ranked[ranked["In weekly queue"]]
hits = int(queue["churn_flag"].sum())

c1, c2, c3, c4 = st.columns(4)
c1.metric("Customers scored", total)
c2.metric("Reviews this week", n)
c3.metric("Lowest churn probability in queue", f"{queue['Churn Probability (%)'].min():.1f}%")
if reveal:
    c4.metric("Actual churn in this week's queue", f"{hits} of {n}")

# ---------- Filters and queue table ----------
st.subheader("Review queue")

f1, f2 = st.columns(2)
regions = sorted(ranked["region"].dropna().unique())
tiers = sorted(ranked["support_tier"].dropna().unique())
selected_regions = f1.multiselect("Region", regions, default=regions)
selected_tiers = f2.multiselect("Support tier", tiers, default=tiers)

search = st.text_input("Search by customer name or ID")
queue_only = st.checkbox("Show only this week's queue", value=True)

filtered = ranked[
    ranked["region"].isin(selected_regions)
    & ranked["support_tier"].isin(selected_tiers)
].copy()

if search:
    term = search.lower()
    filtered = filtered[
        filtered["company_name"].astype(str).str.lower().str.contains(term, na=False)
        | filtered["customer_id"].astype(str).str.lower().str.contains(term, na=False)
    ]

if queue_only:
    filtered = filtered[filtered["In weekly queue"]]

display_cols = [
    "Rank", "customer_id", "company_name", "region", "support_tier",
    "monthly_recurring_revenue", "Churn Probability (%)", "In weekly queue"
]
if reveal:
    display_cols.append("Actual churn (history)")

st.dataframe(filtered[display_cols], width="stretch", hide_index=True)

st.download_button(
    "Download table as CSV",
    filtered[display_cols].to_csv(index=False),
    file_name="review_queue.csv",
    mime="text/csv"
)


# ---------- Reasons (approximate replica of the Logistic Regression) ----------
@st.cache_data
def contributions(data):
    """Each feature's push toward churn, relative to the average customer.
    Uses a scikit-learn Logistic Regression (L2, C=1) on the same five features.
    It approximates the Orange model; it is not the Orange model itself."""
    X = pd.get_dummies(data[NUM + CAT], columns=CAT).astype(float)
    scaler = StandardScaler().fit(X[NUM])
    Xs = X.copy()
    Xs[NUM] = scaler.transform(X[NUM])
    model = LogisticRegression(C=1, max_iter=1000).fit(Xs, data["churn_flag"].astype(int))
    return (Xs - Xs.mean()) * model.coef_[0]


def reasons_for(row_contrib, customer):
    out = []
    for feat, val in row_contrib.sort_values(ascending=False).items():
        if val <= 0 or len(out) == 3:
            break
        if feat == "monthly_recurring_revenue":
            continue  # near-zero effect once other features are in the model
        if feat == "tenure_months":
            out.append(("Tenure", f"{customer['tenure_months']:.0f} months (shorter than typical)", False))
        elif feat == "product_count":
            out.append(("Products", f"{customer['product_count']:.0f} (fewer than typical)", False))
        elif feat.startswith("support_tier_"):
            out.append(("Support tier", feat.replace("support_tier_", ""), True))
        elif feat.startswith("region_"):
            out.append(("Region", feat.replace("region_", ""), True))
    return out


try:
    contrib = contributions(ranked[NUM + CAT + ["churn_flag"]])
except Exception:
    contrib = None

# ---------- Customer details ----------
st.subheader("Customer details")

if "review_log" not in st.session_state:
    st.session_state["review_log"] = []

if not filtered.empty:
    ids = filtered["customer_id"].astype(str).tolist()
    names = dict(zip(filtered["customer_id"].astype(str), filtered["company_name"].astype(str)))
    selected_id = st.selectbox(
        "Select a customer", ids, format_func=lambda x: f"{x} - {names[x]}"
    )
    customer = filtered[filtered["customer_id"].astype(str) == selected_id].iloc[0]

    a, b, c = st.columns(3)
    a.write(f"**Company:** {customer['company_name']}")
    b.write(f"**Region:** {customer['region']}")
    c.write(f"**Support tier:** {customer['support_tier']}")

    a, b, c = st.columns(3)
    a.write(f"**Tenure:** {customer['tenure_months']:.0f} months")
    b.write(f"**Monthly revenue:** ${customer['monthly_recurring_revenue']:,.0f}")
    c.write(f"**Products:** {customer['product_count']:.0f}")

    a, b, c = st.columns(3)
    a.write(f"**Rank:** {int(customer['Rank'])} of {total}")
    b.write(f"**Churn probability:** {customer['Churn Probability (%)']:.1f}%")
    if reveal:
        c.write(f"**Actual churn (history):** {customer['Actual churn (history)']}")
    else:
        c.write(f"**In this week's queue:** {'Yes' if customer['In weekly queue'] else 'No'}")

    st.markdown("**Main factors behind this probability** (approximate)")
    if contrib is not None:
        reasons = reasons_for(contrib.loc[customer.name], customer)
        if reasons:
            for label, value, _static in reasons:
                st.write(f"- {label}: {value}")
            if all(static for _l, _v, static in reasons):
                st.warning(
                    "This probability rests only on region or support tier. Check recent "
                    "tickets and any account-manager contact before reaching out."
                )
        else:
            st.write("No factor stands out above the average customer.")
        st.caption(
            "Factors come from a replica of the Logistic Regression fitted in this app. "
            "They show association in this data, not what causes churn."
        )
    else:
        st.caption("Factors are unavailable for this data.")

    st.markdown("**Review decision**")
    d1, d2 = st.columns([1, 2])
    decision = d1.selectbox("Decision", ["Contact customer", "Monitor", "Dismiss"])
    note = d2.text_input("Note (what did the reviewer check?)")
    if st.button("Save decision"):
        st.session_state["review_log"].append({
            "customer_id": selected_id,
            "company_name": customer["company_name"],
            "decision": decision,
            "note": note,
        })
else:
    st.warning("No customers match the current filters.")

if st.session_state["review_log"]:
    st.markdown("**Review log (this session)**")
    log = pd.DataFrame(st.session_state["review_log"])
    st.dataframe(log, width="stretch", hide_index=True)
    st.download_button(
        "Download review log", log.to_csv(index=False),
        file_name="review_log.csv", mime="text/csv"
    )


# ---------- Who is in this week's queue? ----------
st.subheader("Who is in this week's queue?")
if reveal:
    st.caption(
        "Shows which customers the queue covers, next to how many actually churned in each group."
    )
else:
    st.caption("Shows which regions and support tiers this week's review queue covers.")


def group_table(frame, by):
    rows = []
    for k, g in frame.groupby(by):
        in_queue = int(g["In weekly queue"].sum())
        row = {
            by: k,
            "Customers": len(g),
            "In this week's queue": in_queue,
        }
        if reveal:
            pos = g[g["churn_flag"] == 1]
            caught = int(pos["In weekly queue"].sum())
            row["Actual churn (history)"] = len(pos)
            row["Of those, in queue"] = caught
            row["Share of actual churn in queue (%)"] = (
                round(caught / len(pos) * 100, 1) if len(pos) else None
            )
        rows.append(row)
    return pd.DataFrame(rows)


g1, g2 = st.columns(2)
g1.dataframe(group_table(ranked, "region"), width="stretch", hide_index=True)
g2.dataframe(group_table(ranked, "support_tier"), width="stretch", hide_index=True)

# ---------- Model evaluation (only when outcomes are revealed) ----------
with st.expander("Model evaluation - historical back-test"):
    if not reveal:
        st.write("Tick 'Show actual churn (history)' in the sidebar to see this comparison.")
    else:
        actual = ranked["churn_flag"].astype(int)

        def metrics(flag, show_accuracy=True):
            f = flag.astype(bool)
            tp = int((f & (actual == 1)).sum())
            fp = int((f & (actual == 0)).sum())
            fn = int((~f & (actual == 1)).sum())
            tn = int((~f & (actual == 0)).sum())
            recall = tp / (tp + fn) if (tp + fn) else None
            precision = tp / (tp + fp) if (tp + fp) else None
            accuracy = (tp + tn) / len(actual)
            fmt = lambda x: f"{x:.1%}" if x is not None else "n/a"
            return {
                "Customers flagged": int(f.sum()),
                "Recall": fmt(recall),
                "Precision": fmt(precision),
                "Accuracy": fmt(accuracy) if show_accuracy else "-",
            }

        rows = {
            "Predict no one churns (baseline)": metrics(pd.Series(False, index=ranked.index)),
            "Flag every Basic-tier customer (rule)": metrics(ranked["support_tier"] == "Basic"),
            "Logistic Regression at 0.5": metrics(ranked[PRED] == 1),
            f"This week's queue (top {n})": metrics(ranked["In weekly queue"], show_accuracy=False),
        }
        st.dataframe(pd.DataFrame(rows).T, width="stretch")
        cap = min(n, total_pos) / total_pos if total_pos else 0
        st.caption(
            "Recall = share of all customers who actually churned that were flagged. "
            "Precision = share of flagged customers who actually churned. "
            f"The weekly queue flags only {n} customers, so its recall cannot exceed {cap:.0%}; "
            "compare its precision, not its recall, with the other rows. "
            "Accuracy alone is misleading: predicting that no one churns already scores about 75%. "
            "'Actual churn' is the historical label, which includes customers flagged as at-risk. "
            "These are out-of-fold results on historical data, not a guarantee of future performance."
        )

Writing app.py


In [ ]:
!pip -q install streamlit pyngrok

from pyngrok import ngrok

ngrok.set_auth_token("")

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://size-kilometer-system.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0



2026-10-23 06:21:49.080 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.240.145.147:8501



  Stopping...
  Stopping...
